# 12 · Embedding 高级主题

> 把向量化做到“生产级”需要掌握的进阶细节：查询与文档的不对称、维度与归一化、多语言、以及可裁剪维度。

**本文件覆盖知识点**：Query Embedding / Document Embedding / Query-Document Prefix / Matryoshka Embedding / Dimension / Normalization / Multilingual / Domain-specific / 微调

In [ ]:
# ===== 本课共用：真调 LLM 做「说明 / 演示」的小助手 =====
# 凡某个知识点能靠“真调一次大模型”当场讲清 / 演示的，下面的 cell 都用 _llm_live()
# 真调 qwen-plus 并打印模型输出作为说明；只有在项目根 .env 配了 DASHSCOPE_API_KEY 时才真调，
# 没配置就打印一段固定的演示样例，保证整个 notebook 不联网也能完整读下来。
from dotenv import load_dotenv; load_dotenv()
import os
from dashscope import Generation

_KEY = os.getenv('DASHSCOPE_API_KEY', '').strip()
_HAS_KEY = bool(_KEY) and '你的' not in _KEY

def _llm_live(prompt, fallback, system='你是资深 RAG 讲师，回答精炼、结构清晰、尽量结合例子。', temperature=0.3, model='qwen-plus'):
    """真调一次 qwen-plus 并打印结果；无 Key 时打印 fallback 作为演示样例。返回模型文本或 None。"""
    if not _HAS_KEY:
        print('未在 .env 配置 DASHSCOPE_API_KEY，跳过实时调用。以下是固定演示样例（配置后自动变为实时输出）：')
        print(fallback)
        return None
    msgs = [{'role': 'system', 'content': system}, {'role': 'user', 'content': prompt}]
    try:
        r = Generation.call(model=model, messages=msgs, temperature=temperature, result_format='message', api_key=_KEY)
        if r.status_code == 200:
            text = r.output.choices[0].message.content
            print('—— 模型实时输出 ——')
            print(text)
            return text
        print('调用失败：', getattr(r, 'code', ''), getattr(r, 'message', ''))
    except Exception as e:
        print('调用异常：', e)
    print('fallback：')
    print(fallback)
    return None


## 1. Query / Document Prefix：检索为什么常常“不对称”

检索任务里“问句”与“资料”风格完全不同：问句短而口语（“怎么收费？”），资料长而书面（“付费套餐分为…”）。
许多模型为此给两类文本加不同的前缀指令，拉近它们与指令的语义锚点：

```text
查询  →  "为这个搜索查询生成表示: "  + 问题
文档  →  "为这个文档生成表示: "    + 正文
```

> 工程要点：查 API 文档确认模型是否要求前缀；查询与文档必须用同一条向量化通道的对应模式，否则相似度失真。

In [ ]:
# 知识点·真调说明：Query/Document 不对称 —— 让模型把同一意图写成“口语问句”与“书面文档句”，看检索为何要分开处理
print('① 同一检索意图，两种文本形态：口语问句 vs 书面文档表述')
_llm_live(
    prompt='请就同一个用户意图“这个产品怎么收费”，分别写两段：\n'
           '1) 一段用户会打出的“口语问句”（5~12 字）；\n'
           '2) 一段知识库文档里的“书面表述”（40 字上下，像产品手册的一句话，要自洽、能独立成立）。',
    system='你是资深 RAG 讲师。要求：只输出两段，先标“问句：”再标“文档：”，中间空一行，不要多余解释。',
    fallback='未配置 Key 的固定样例：\n'
             '问句：你们怎么收费？按年还是按月？\n'
             '文档：本产品采用订阅制计费，提供按月与按年两种套餐，企业版支持按量定制报价。',
    temperature=0.3,
)
print()
print('② 让模型点评：两边丢进同一个向量化流程、不加区分，最常在什么情况下失真')
_llm_live(
    prompt='很多向量检索模型要求“查询”和“文档”用不同的指令前缀，例如查询前加“为这个搜索查询生成表示”。'
           '请用不超过 3 句话解释：为什么建议区分处理？不区分的话，相似度检索最容易在什么场景下失真？'
           '并给出一个具体例子。',
    system='你是资深 RAG 讲师。要求：简短、给具体例子，不讲数学。',
    fallback='未配置 Key 的固定样例：\n'
             '问句短而口语、文档长而书面，两者直接走同一表示会让向量落在不同的分布区域，'
             '问句容易命中“语气像”而不是“含义像”的文本；给两边各自的前缀指令，是把它们对齐到同一条语义标尺上，'
             '相似度才可比。例如口语问句“能白嫖吗”若不改写，很难与文档里“免费试用”这一段对齐。',
    temperature=0.2,
)
print()
print('→ 工程要点：先查向量服务文档确认是否要前缀；查询与文档务必走“同一通道的对应模式”，否则相似度失真。')

In [ ]:
# Dimension + Normalization 是两个直接影响系统设计的参数
print('''
1) 维度(Dimension)
   - 维度越高，信息容量越大，但存储/算力越高
   - 有些服务允许输出较粗的"截断维度"以省钱

2) 归一化(Normalization)
   - L2 归一化后，点积=余弦，向量库可用内积索引(更快)
   - 写入库的向量务必与检索时同一归一化规则

3) Matryoshka 思路
   - 训练时让向量的"前缀子向量"也能独立可用
   - 需要时可以只取前 N 维作为近似表示，动态权衡精度/成本
''')

import numpy as np
def l2_normalize(a):
    return a / (np.linalg.norm(a) + 1e-10)

vec = np.array([3.0, 4.0])
print('原始 [3,4] 模长=5，归一化后:', np.round(l2_normalize(vec), 3), '模长≈', round(np.linalg.norm(l2_normalize(vec)), 3))

## 2. Multilingual / 中文 / Domain-specific

- **多语言**：跨语言问答需要“对齐空间”，即同一语义的不同语言落在相近位置。BGE-M3、GTE 等支持多语言；
- **中文**：中文按 token/字粒度不同，评测要跑中文语料，别直接套用英文榜单；
- **领域专属**：代码用代码嵌入、生物用生物文本嵌入；通用模型在专有词上区分度差时换领域模型。

In [ ]:
# 知识点·真调说明：Multilingual / 中文 / 领域选型 —— 把“何时该换 embedding 模型”变成一次可照抄的决策问询
_llm_live(
    prompt='我在为一个“中文医疗器械维修问答 RAG”选向量模型：语料是中文维修手册、含大量厂商型号与专有词，'
           '用户提问以中文为主、偶尔夹英文型号。\n'
           '请按三步给出建议（每步 1~2 句）：'
           '① 先评估该用“通用多语言模型”还是“中文/领域专用模型”，说明理由；'
           '② 换模型之前，先用哪些“现象”判断当前通用模型不够用（给 2 条可落地检验）；'
           '③ 确认要换时，如何低成本验证新模型在专名词库上确实更优。',
    system='你是 RAG 向量模型选型顾问。要求分①②③、语言精炼、可落地，不要罗列广告。',
    fallback='未配置 Key 的固定样例：\n'
             '① 通常先用通用多语言模型打底：语料以中文为主、英文型号只占少量时，'
             '统一向量空间的收益往往大于单语模型，且生态与评测资料更全。\n'
             '② 两条可落地检验：把同一专名/型号的不同写法两两算相似度，若同名不同写得分反而低于无关文本，'
             '说明区分度不足；再拿少量真实问答对看 Top-1 命中率。\n'
             '③ 换前先做 200 条领域问答对的小样本对比，只看“专有词召回与排序”两类错误率，再决定是否值得迁移。',
    temperature=0.2,
)
print('→ 中文评测不能照搬英文榜单（§2 的提醒）；领域差要先“量化现象”再决定换模型，而不是默认追新。')

## 3. Embedding 微调：对比学习一句话

目标：把正例对（问，答）拉近、把负例对（问，错答）推远。

```text
loss = -log( e^{sim(q,d+)} / Σ_{d'∈{d+,d-}} e^{sim(q,d')} )    # InfoNCE 对比损失
```
- 需要数据：一批 (q, d+, d-) 三元组；难负例(hard negative)收益大；
- 完整实操思路在第 39 课。



In [ ]:
# 知识点·真调说明：Embedding 微调 / 难负例 —— 让模型现场生成“正例 vs 难负例”，看清难负例难在哪
_llm_live(
    prompt='我要用对比学习微调一个“笔记本售后问答”的 embedding 模型。检索问题 q = “笔记本插电后风扇一直狂转怎么办”。'
           '请分别写一段 30 字上下的文本：\n'
           '① 正例 d+：真正回答该问题、应被拉近的文档片段；\n'
           '② 难负例 d-：主题词很像、看着“相关”，但答非所问或结论错的片段（用来考验模型能否分辨细节）；\n'
           '③ 易负例：与主题无关的一段。\n'
           '最后用一句话说明：为什么 ② 比 ③ 对训练更有价值。',
    system='你是负责构造 embedding 训练语料的工程师。要求先按①②③分段输出文本，再单独一行给出“为什么难负例更有价值”，不写训练代码。',
    fallback='未配置 Key 的固定样例：\n'
             '① 正例：笔记本风扇狂转多由温度过高或灰尘堵塞引起，可在电源设置中限制 CPU 睿频并清理出风口。\n'
             '② 难负例：台式机机箱风扇噪音大，可换更大尺寸风扇降噪。——它同样是“风扇/噪音”主题，'
             '但答案针对台式机；模型若只靠表面词重合就给高分，说明没学到“问题-答案”的真实对应。\n'
             '③ 易负例：今天晴天，适合晾晒被子。\n'
             '难负例逼模型放弃“看词不看义”的捷径、去学真正的问答匹配，带来的区分度远大于随便拉来的无关负例。',
    temperature=0.2,
)
print('→ “用 LLM 造难负例/合成问答对”正是对比学习微调的前置动作，完整实操思路见第 39 课。')

## 小结

- 查询/文档加指令前缀解决不对称；
- 统一归一化规则，理解维度对成本的影响；
- 多语言/中文/领域按语料选择；专有领域差再走对比学习微调。